# Crabtree Creek: predicción de crecientes

Johann Alexander Vera Mercado. Cinco años de datos USGS, 5 minutos, anticipación de 1–6 horas.

Esta versión predice elevación en pies NAVD88. No incluye caudal. Conserva picos y evalúa períodos separados. Leer README.md y GUIA_ES.md antes de interpretar alertas.

## 1. Preparación
Ejecutar desde la carpeta del proyecto. En Colab se puede cargar y extraer la copia ZIP completa o clonar el repositorio público. Los comandos siguientes no requieren acceso a Google Drive.

In [ ]:
!pip install -r requirements.txt


In [ ]:
!python download_data.py
!python -m unittest test_integrity.py


## 2. Revisar datos sin borrar picos
Los faltantes se conservan. Las fechas se convierten a UTC, y los solapamientos se eliminan. El código no sustituye lluvia faltante por cero.

In [ ]:
from forecast import load, features
df, audit = load()
print(audit)
df[["level_ft", "rain_in"]].plot(subplots=True, figsize=(12, 6))


## 3. Entrenamiento
Entrenamiento: hasta octubre 2024; validación: octubre 2024–2025; prueba: desde octubre 2025. Se excluyen objetivos que cruzan los límites. Los niveles altos reciben peso 4. Se selecciona la configuración con validación, no con prueba.

In [ ]:
!OMP_NUM_THREADS=2 python forecast.py
!python evaluate_events.py


In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.read_csv("results/metrics.csv"))
display(pd.read_csv("results/threshold_metrics.csv"))
display(Image("results/largest_test_event.png"))


## 4. Interpretación hidrológica
Revisar magnitud y momento del pico, y eventos omitidos. El umbral de entrenamiento es estadístico, no operativo. La gráfica muestra pronósticos sucesivos alineados por hora válida, no una trayectoria emitida una sola vez. A 4–6 horas el modelo no supera persistencia en error global.

Confirmar la semántica del sensor de lluvia antes de interpretar sumas como acumulaciones físicas.

## 5. Redes neuronales opcionales
Código corregido para continuar la comparación original; NO entrenado en esta primera ejecución. Requiere TensorFlow. Para comparar modelos, alinear los timestamps elegibles.

In [ ]:
# Ejecutar opcionalmente:
# !pip install -r requirements-neural.txt
# !python neural_models.py --kind gru --epochs 30
